In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [2]:
#подгружаем дата файлы и делаем объединение
event = pd.read_csv('event.csv')
shift = pd.read_csv('shift.csv')
user = pd.read_csv('user.csv')

In [3]:
event['ts'] = pd.to_datetime(event['ts'])
shift['start_at'] = pd.to_datetime(shift['start_at'])

In [4]:
event.shape

(383883, 5)

In [5]:
event.head(5)

,id,shift_id,user_id,interaction,ts
0,ab450739fc9176a3ba7e92e6bbc9b169,12396,69359158a2c25806c20b11f860fa46ab,VIEW,2026-01-06
1,46fd04be38ec4738278ec9e941463802,12396,258962a7b6a7e68cadaaf0a927eadc90,VIEW,2026-01-05
2,8ca0102b2784c383dc430f795b6f5377,12396,258962a7b6a7e68cadaaf0a927eadc90,VIEW,2026-01-03
3,e3004221404186d95b086dc0cb688905,12396,258962a7b6a7e68cadaaf0a927eadc90,VIEW,2026-01-07
4,e5be648f4c7d19f33cbc06409f8525f5,12396,984483ffe4b8fd56a7d9d78e9fe97627,VIEW,2026-01-07


In [6]:
shift.head(5)

,id,start_at,location_id,task_type,employer_id,workplace_id,need_mk,id_differential,hours,reward,capacity
0,9857,2026-01-06 12:30:00+00:00,161,Погрузка и разгрузка товара,c1714b6178cdbe470384bdd7ea181f3c,27b794bbdb945cc982c40394dc9b520a,False,False,3,900.0,2
1,21761,2026-01-15 12:30:00+00:00,161,Погрузка и разгрузка товара,c1714b6178cdbe470384bdd7ea181f3c,27b794bbdb945cc982c40394dc9b520a,False,False,3,900.0,2
2,21647,2026-01-24 12:30:00+00:00,161,Погрузка и разгрузка товара,c1714b6178cdbe470384bdd7ea181f3c,27b794bbdb945cc982c40394dc9b520a,False,False,3,900.0,2
3,38993,2026-01-06 09:00:00+00:00,13,Помощь в торговом зале,e0480c3c5525389f776aa4fa58a7f72b,b74ac875bc93d4c4ae24893c99147640,True,False,7,1800.0,1
4,48240,2026-01-04 16:00:00+00:00,13,Помощь в торговом зале,e0480c3c5525389f776aa4fa58a7f72b,b74ac875bc93d4c4ae24893c99147640,True,False,7,1800.0,1


In [7]:
user.head(5)

,location_id,is_strict_location,id,has_mk
0,64,False,0a66722f9259c08a6a6d78d2ea90e375,True
1,61,False,314a2bc9a488ba22a31ad0d9a0c1c26f,True
2,181,False,73f5bbba685e5e676f59a4709f04c825,True
3,234,False,50dd1bf4e4ba7ebec4a45076be1dc90c,True
4,158,False,c53805ed7cbcabe72f70da8404213552,True


In [8]:
event_full = (event
    .merge(shift, left_on='shift_id', right_on='id', how='left')
    .merge(user, left_on='user_id', right_on='id', how='left')
    .drop(columns=['id_x', 'id_y'], errors='ignore'))

In [9]:
event_full.tail(5)

,shift_id,user_id,interaction,ts,start_at,location_id_x,task_type,employer_id,workplace_id,need_mk,id_differential,hours,reward,capacity,location_id_y,is_strict_location,id,has_mk
383878,46323,d13ec6d546a7cbc0ea091df042489fa0,FINISHED,2026-01-28,2026-01-28 13:00:00+00:00,64,Выкладка товара,fb3d1aaa7ddb54fd11604d5529a7774b,b816f1cd6a79d60f32cb89c9f5eb7028,True,False,8,2700.0,1,64,False,d13ec6d546a7cbc0ea091df042489fa0,True
383879,46323,af40af56f60b3fbc8e5aadb4fc687f11,SYSTEM_CANCEL,2026-01-25,2026-01-28 13:00:00+00:00,64,Выкладка товара,fb3d1aaa7ddb54fd11604d5529a7774b,b816f1cd6a79d60f32cb89c9f5eb7028,True,False,8,2700.0,1,64,False,af40af56f60b3fbc8e5aadb4fc687f11,True
383880,32605,67b8d8989c0a30cef2d33b67b837208e,FINISHED,2026-01-30,2026-01-30 08:00:00+00:00,83,Выкладка товара,fb3d1aaa7ddb54fd11604d5529a7774b,d379b233a644b30c3dd76d747f59b1b1,True,False,10,2800.0,1,83,False,67b8d8989c0a30cef2d33b67b837208e,True
383881,53015,c036e3971821db4ae537f9c3a8ddc2cb,SYSTEM_CANCEL,2026-01-28,2026-01-29 15:00:00+00:00,121,Выкладка товара,fb3d1aaa7ddb54fd11604d5529a7774b,f9a9a9803adf8d62e59f05679b7e4326,True,False,7,2100.0,1,121,False,c036e3971821db4ae537f9c3a8ddc2cb,True
383882,53015,e34a200bc4a167d4b39fa2a5de6f296f,FINISHED,2026-01-29,2026-01-29 15:00:00+00:00,121,Выкладка товара,fb3d1aaa7ddb54fd11604d5529a7774b,f9a9a9803adf8d62e59f05679b7e4326,True,False,7,2100.0,1,121,False,e34a200bc4a167d4b39fa2a5de6f296f,True


In [10]:
event_full['interaction'].value_counts()

,count
interaction,
VIEW,318747
APPLY,24030
FINISHED,17305
SYSTEM_CANCEL,12998
USER_CANCEL,10803


In [11]:
#сортируем данные по времени и получаем временную метку (делаем подготовку к разбиению данных)
event_full = event_full.sort_values('ts')
unique_dates = event_full['ts'].unique()

In [12]:
#проверим периоды через TimeSeriesSplit
n_splits = 3          # на сколько частей делим данные (3 итерации)
test_size = 21        # валидация = 2 недели (14 дней)
gap = 0
tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=gap)
fold = 1
for train_indices, test_indices in tscv.split(unique_dates):
      train_dates = unique_dates[train_indices]
      test_dates = unique_dates[test_indices]
      print(f"\n=== Итерация {fold} ===")
      print(f"  Train периоды: с {train_dates[0]} по {train_dates[-1]}")
      print(f"  Test  периоды: с {test_dates[0]} по {test_dates[-1]}")
      fold += 1


=== Итерация 1 ===
  Train периоды: с 2025-12-16 00:00:00 по 2025-12-27 00:00:00
  Test  периоды: с 2025-12-28 00:00:00 по 2026-01-17 00:00:00

=== Итерация 2 ===
  Train периоды: с 2025-12-16 00:00:00 по 2026-01-17 00:00:00
  Test  периоды: с 2026-01-18 00:00:00 по 2026-02-07 00:00:00

=== Итерация 3 ===
  Train периоды: с 2025-12-16 00:00:00 по 2026-02-07 00:00:00
  Test  периоды: с 2026-02-08 00:00:00 по 2026-02-28 00:00:00


In [13]:
#выбираем split_date = '2026-02-14' по последней итерации окна
split_date = '2026-02-07'
#делаем разделение выборки
train_events = event_full[event_full['ts'] <= split_date].copy()
test_events = event_full[event_full['ts'] > split_date].copy()

In [14]:
train_events['interaction'].value_counts()

,count
interaction,
VIEW,318060
APPLY,23721
FINISHED,17082
SYSTEM_CANCEL,12866
USER_CANCEL,10693


In [15]:
#подсчитаем кол-во новыйх пользователей в тесте по сравнению с трейном
test_users = set(event_full[event_full['ts'] > split_date ]['user_id'].unique())
train_users = set(event_full[event_full['ts'] <= split_date ]['user_id'].unique())
new_users = test_users - train_users

print(f"Новых пользователей в test: {len(new_users)}")

Новых пользователей в test: 42


In [16]:
#создаем целевую переменную
def create_target(events):
     # FINISHED = 1
    finished = events[events['interaction'] == 'FINISHED'][['user_id', 'shift_id']].drop_duplicates()
     # APPLY без USER_CANCEL = 1
    apply_pairs = events[events['interaction'] == 'APPLY'][['user_id', 'shift_id']].drop_duplicates()
    cancel_pairs = events[events['interaction'] == 'USER_CANCEL'][['user_id', 'shift_id']].drop_duplicates()
    apply_clean = apply_pairs.merge(cancel_pairs.assign(is_cancel=1), on=['user_id', 'shift_id'], how='left')
    apply_clean = apply_clean[apply_clean['is_cancel'].isna()][['user_id', 'shift_id']]
    #Объединяем все положительные
    positive = pd.concat([finished, apply_clean], ignore_index=True)
    positive = positive.drop_duplicates()
    positive['target'] = 1
    #Создаем результат (target=0 по умолчанию через fillna)
    all_pairs = events[['user_id', 'shift_id']].drop_duplicates()
    result = all_pairs.merge(positive, on=['user_id', 'shift_id'], how='left')
    result['target'] = result['target'].fillna(0).astype(int)
    # Добавляем время события (берем максимальное для пары)
    event_times = events.groupby(['user_id', 'shift_id'])['ts'].max().reset_index()
    result = result.merge(event_times, on=['user_id', 'shift_id'], how='left')
    return result[['user_id', 'shift_id', 'ts', 'target']]

In [17]:
train_target = create_target(train_events)
test_target = create_target(test_events)

In [18]:
train_target['target'].value_counts()


,count
target,
0,247500
1,26784


In [19]:
train_target.head(3)

,user_id,shift_id,ts,target
0,6e36ad7287c060819e59f85d523febb5,2665,2026-01-02,0
1,6fbbdde6bb746e3b5279b857f3146c37,7082,2025-12-17,0
2,a083a2c9213016c0114a351dd56f9056,2132,2025-12-18,0


In [20]:
#функция для создания признаков
def create_features(events, target_pairs):

    # Берем признаки смен (из shift)
    shift_features = events[['shift_id', 'start_at', 'location_id_x', 'task_type',
                              'employer_id', 'workplace_id', 'need_mk',
                              'id_differential', 'hours', 'reward', 'capacity']].drop_duplicates('shift_id')
    shift_features = shift_features.rename(columns={'location_id_x': 'shift_location'})

    # Берем признаки пользователей
    user_features = events[['user_id', 'location_id_y', 'is_strict_location', 'has_mk']].drop_duplicates('user_id')
    user_features = user_features.rename(columns={'location_id_y': 'user_location'})
    result = target_pairs.merge(shift_features, on='shift_id', how='left')
    result = result.merge(user_features, on='user_id', how='left')
    # Общая активность пользователя
    user_stats = []
    for user_id in result['user_id'].unique():
        user_events = events[events['user_id'] == user_id]
        total_views = len(user_events[user_events['interaction'] == 'VIEW'])
        total_applies = len(user_events[user_events['interaction'] == 'APPLY'])
        total_finished = len(user_events[user_events['interaction'] == 'FINISHED'])
        total_cancels = len(user_events[user_events['interaction'] == 'USER_CANCEL'])
        total_system_cancels = len(user_events[user_events['interaction'] == 'SYSTEM_CANCEL'])
        # Конверсии и метрики
        apply_rate = total_applies / total_views if total_views > 0 else 0
        success_rate = total_finished / total_applies if total_applies > 0 else 0
        cancel_rate = total_cancels / total_applies if total_applies > 0 else 0

        # Дополнительные метрики
        finished_rate = total_finished / total_views if total_views > 0 else 0
        reliability = (total_finished - total_cancels) / total_applies if total_applies > 0 else 0

        user_stats.append({
            'user_id': user_id,
            'user_total_views': total_views,
            'user_total_applies': total_applies,
            'user_total_finished': total_finished,
            'user_total_cancels': total_cancels,
            'user_apply_rate': apply_rate,
            'user_success_rate': success_rate,
            'user_cancel_rate': cancel_rate,
            'user_finished_rate': finished_rate,
            'user_reliability': reliability})
    user_stats_df = pd.DataFrame(user_stats)

    # Объединяем
    result = result.merge(user_stats_df, on='user_id', how='left')

    # Добавляем вычисляемые признаки
    result['start_datetime'] = pd.to_datetime(result['start_at']) #дата начала смены
    result['shift_hour'] = result['start_datetime'].dt.hour #время начала смены
    result['shift_dayofweek'] = result['start_datetime'].dt.dayofweek #день недели начала смены
    result['shift_is_weekend'] = (result['shift_dayofweek'] >= 5).astype(int) #метка выходные или рабочие
    result['reward_per_hour'] = result['reward'] / result['hours'] #оплата часа
    result['location_match'] = (result['user_location'] == result['shift_location']).astype(int) #совпадает ли локация смены и пользователя
    result['mk_compatible'] = ((result['need_mk'] == 0) | (result['has_mk'] == 1)).astype(int) #совпадение по медкнижке

    #доп признак на время принятия решения и время принятия смены
    first_views = events[events['interaction'] == 'VIEW'].groupby(['user_id', 'shift_id'])['ts'].min().reset_index()
    first_views.columns = ['user_id', 'shift_id', 'first_view_time']
    applies = events[events['interaction'] == 'APPLY'][['user_id', 'shift_id', 'ts']].drop_duplicates(subset=['user_id', 'shift_id'])
    applies.columns = ['user_id', 'shift_id', 'apply_time']
    result = result.merge(first_views, on=['user_id', 'shift_id'], how='left')
    result = result.merge(applies, on=['user_id', 'shift_id'], how='left')
    result['decision_hours'] = (result['apply_time'] - result['first_view_time']).dt.total_seconds()/3600
    result['decision_hours'] = result['decision_hours'].fillna(0)
    result['decision_hours'] = result['decision_hours'].clip(lower=0)
    result = result.drop(['first_view_time', 'start_datetime','apply_time'], axis=1)

    #capacity спец признаки для метрики топ-10 по ROC_AUC
    result['capacity_log'] = np.log1p(result['capacity']) #Логарифм capacity (сглаживает влияние больших значений)
    result['capacity_ratio_to_10'] = result['capacity'] / 10 #Отношение capacity к 10 (для понимания лимита FPR)
    result['needed_candidates'] = result['capacity'].clip(upper=10) #Признак: сколько кандидатов нужно набрать (capacity или 10)

    return result


In [21]:
train_features = create_features(train_events, train_target)
test_features = create_features(test_events, test_target)

In [22]:
train_features.head(5)

,user_id,shift_id,ts,target,start_at,shift_location,task_type,employer_id,workplace_id,need_mk,...,shift_hour,shift_dayofweek,shift_is_weekend,reward_per_hour,location_match,mk_compatible,decision_hours,capacity_log,capacity_ratio_to_10,needed_candidates
0,6e36ad7287c060819e59f85d523febb5,2665,2026-01-02,0,2026-01-04 08:00:00+00:00,282,Фасовка готовой продукции,b27ed42a7844a46e515ce7f38cd92e41,b397cd200ad03575782561c396af3db0,True,...,8,6,1,300.000000,0,1,0.0,2.397895,1.0,10
1,6fbbdde6bb746e3b5279b857f3146c37,7082,2025-12-17,0,2026-01-01 17:00:00+00:00,308,Помощь на кухне,79aa54b4b7886b3d3ef45a589198138e,4e87d14ba1172e1fe97f35c0fa05f392,True,...,17,3,0,283.333333,1,1,0.0,0.693147,0.1,1
2,a083a2c9213016c0114a351dd56f9056,2132,2025-12-18,0,2026-01-02 07:00:00+00:00,121,Мойка посуды и инвентаря,02d2b372ccc7404c03bf705d67360723,7edd370032d71765ea1470168770e739,True,...,7,4,0,266.666667,1,1,0.0,0.693147,0.1,1
3,2b4e28c9fe9b6235eb257605512d97b2,5628,2025-12-18,0,2026-01-01 07:00:00+00:00,121,Мойка посуды и инвентаря,02d2b372ccc7404c03bf705d67360723,7edd370032d71765ea1470168770e739,True,...,7,3,0,266.666667,1,1,0.0,0.693147,0.1,1
4,a083a2c9213016c0114a351dd56f9056,5628,2025-12-18,0,2026-01-01 07:00:00+00:00,121,Мойка посуды и инвентаря,02d2b372ccc7404c03bf705d67360723,7edd370032d71765ea1470168770e739,True,...,7,3,0,266.666667,1,1,0.0,0.693147,0.1,1


In [23]:
#подготавливаем данные для обучения
exclude_cols = ['user_id', 'shift_id', 'ts', 'target', 'start_at', 'task_type', 'employer_id', 'workplace_id']
feature_cols = [col for col in train_features.columns if col not in exclude_cols]
# Разделяем на X и y
X_train = train_features[feature_cols]
y_train = train_features['target']
X_test = test_features[feature_cols]
y_test = test_features['target']
#разделим train на валидацию и обучение
X_train_part, X_val, y_train_part, y_val = train_test_split(X_train, y_train,test_size=0.2,random_state=42,stratify=y_train)

In [24]:
model = lgb.LGBMClassifier(
    n_estimators=150,
    max_depth=7,
    num_leaves=31,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_lambda=0.1,
    reg_alpha=0.1,
    class_weight='balanced',  # для учета дисбаланса классов
    n_jobs=1,
    random_state=42,
    verbose=-1)

model.fit(
    X_train_part, y_train_part,
    eval_set=[(X_train_part, y_train_part), (X_val, y_val)],
    eval_metric=['auc', 'average_precision'],
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(50)])

Training until validation scores don't improve for 30 rounds
[50]	training's auc: 0.853157	training's average_precision: 0.859158	training's binary_logloss: 0.474765	valid_1's auc: 0.84681	valid_1's average_precision: 0.498689	valid_1's binary_logloss: 0.475272
[100]	training's auc: 0.865704	training's average_precision: 0.869265	training's binary_logloss: 0.456314	valid_1's auc: 0.854317	valid_1's average_precision: 0.510498	valid_1's binary_logloss: 0.462097
[150]	training's auc: 0.874485	training's average_precision: 0.876565	training's binary_logloss: 0.443538	valid_1's auc: 0.858626	valid_1's average_precision: 0.518851	valid_1's binary_logloss: 0.453206
Did not meet early stopping. Best iteration is:
[150]	training's auc: 0.874485	training's average_precision: 0.876565	training's binary_logloss: 0.443538	valid_1's auc: 0.858626	valid_1's average_precision: 0.518851	valid_1's binary_logloss: 0.453206


LGBMClassifier(class_weight='balanced', colsample_bytree=0.8, max_depth=7,
               n_estimators=150, n_jobs=1, random_state=42, reg_alpha=0.1,
               reg_lambda=0.1, subsample=0.8, verbose=-1)

In [25]:
y_pred_test = model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_test)
print(f"Test AUC: {test_auc:.4f}")

Test AUC: 0.8925


In [26]:
# Сравните train и test AUC
train_pred = model.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, train_pred)
print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC:  {test_auc:.4f}")
print(f"Разница:   {train_auc - test_auc:.4f}")

# Если разница < 0.05 - переобучения нет
# У вас, скорее всего, будет небольшая разница

Train AUC: 0.8713
Test AUC:  0.8925
Разница:   -0.0212


In [27]:
#проверяем ранжирует ли модель кандидатов по вероятности попадания в смену
def validate_top10_predictions(model, features, test_data, shift_capacity_dict):
    # Группируем по сменам
    results = []
    for shift_id in features['shift_id'].unique():
        shift_mask = features['shift_id'] == shift_id
        shift_features = features.loc[shift_mask]
        shift_target = test_data.loc[shift_mask, 'target'] if 'target' in test_data.columns else None
        shift_capacity = shift_capacity_dict.get(shift_id, 10)

        # Получаем предсказания
        if len(shift_features) > 0:
            scores = model.predict_proba(shift_features[feature_cols])[:, 1]

            # Берем ТОП-10 (ровно 10 кандидатов)
            n_candidates = min(10, len(scores))
            top_indices = np.argsort(scores)[-n_candidates:][::-1]
            top_scores = scores[top_indices]

            # Проверяем, что вернули не больше 10
            assert len(top_indices) <= 10, f"Вернуто больше 10 кандидатов: {len(top_indices)}"

            # Проверяем, что ранжирование корректное
            if len(top_indices) > 1:
                assert np.all(np.diff(top_scores) <= 0), "Ранжирование нарушено!"

            results.append({
                'shift_id': shift_id,
                'capacity': shift_capacity,
                'n_candidates': len(top_indices),
                'top_scores': top_scores,
                'max_fpr': min(1.0, shift_capacity / 10)})

    print("Проверка топ-10:")
    print(f"  Всего смен: {len(results)}")
    print(f"  Смен с 10 кандидатами: {sum(r['n_candidates'] == 10 for r in results)}")
    print(f"  Смен с <10 кандидатами: {sum(r['n_candidates'] < 10 for r in results)}")

    return results

# Проверяем
shift_capacities = dict(zip(test_features['shift_id'], test_features['capacity']))
top10_results = validate_top10_predictions(model, test_features, test_target, shift_capacities)

Проверка топ-10:
  Всего смен: 239
  Смен с 10 кандидатами: 10
  Смен с <10 кандидатами: 229


In [32]:
# Расчет метрики с ограничением по FPR для каждого дня и capacity
def calculate_restricted_auc_by_day(test_features, y_pred, test_date='2026-02-15'):
    test_features = test_features.copy()
    test_features['pred_score'] = y_pred
    test_features['date'] = pd.to_datetime(test_features['ts']).dt.date

    # Группируем по дням и capacity
    daily_results = []
    for date in test_features['date'].unique():
        day_data = test_features[test_features['date'] == date]

        # Группируем по capacity
        capacity_groups = day_data.groupby('capacity')
        day_aucs = []
        day_info = []
        for capacity, group in capacity_groups:
          if len(group) == 0:
                continue
          n_positive = group['target'].sum()
          n_negative = len(group) - n_positive
            # Проверяем, есть ли оба класса
          if n_positive > 0 and n_negative > 0:
                max_fpr = min(1.0, capacity / 10)

                try:
                    auc_restricted = roc_auc_score(
                        group['target'],
                        group['pred_score'],
                        max_fpr=max_fpr)
                    day_aucs.append(auc_restricted)
                    day_info.append({
                        'capacity': capacity,
                        'auc': auc_restricted,
                        'n_positive': n_positive,
                        'n_negative': n_negative,
                        'max_fpr': max_fpr})
                except Exception as e:
                    print(f"  Ошибка для {date}, capacity={capacity}: {e}")
                    day_aucs.append(np.nan)
          else:
                # Только один класс - пропускаем эту группу
                print(f"  Пропущена группа {date}, capacity={capacity}: только {n_positive} positive, {n_negative} negative")
                day_aucs.append(np.nan)


       # Считаем среднее только по не-nan значениям
        valid_aucs = [a for a in day_aucs if not np.isnan(a)]

        if valid_aucs:
            daily_metric = np.mean(valid_aucs)
            daily_results.append({
                'date': date,
                'metric': daily_metric,
                'n_groups': len(valid_aucs),
                'total_groups': len(day_aucs),
                'groups_info': day_info})

    print("\nПроверка метрики по дням:")
    valid_metrics = []
    for res in daily_results:
        print(f"\n{res['date']}:")
        print(f"  Метрика дня: {res['metric']:.4f}")
        print(f"  Групп учтено: {res['n_groups']} из {res['total_groups']}")

        # Показываем по каждой capacity
        for info in res['groups_info']:
            print(f"    capacity={info['capacity']}: AUC={info['auc']:.4f} "
                  f"(pos={info['n_positive']}, neg={info['n_negative']}, "
                  f"max_fpr={info['max_fpr']:.2f})")

        valid_metrics.append(res['metric'])

    # Средняя метрика
    if valid_metrics:
        avg_metric = np.mean(valid_metrics)
        print(f"\n{'='*60}")
        print(f"СРЕДНЯЯ МЕТРИКА ПО ДНЯМ: {avg_metric:.4f}")
        print(f"Учтено дней: {len(valid_metrics)} из {len(test_features['date'].unique())}")
        print(f"{'='*60}")
    else:
        avg_metric = np.nan
        print("\nНет дней с валидной метрикой!")

    return daily_results, avg_metric

# Применяем
daily_metrics, avg_metric = calculate_restricted_auc_by_day(test_features, y_pred_test)

  Пропущена группа 2026-02-08, capacity=4: только 0 positive, 2 negative
  Пропущена группа 2026-02-18, capacity=4: только 16 positive, 0 negative
  Пропущена группа 2026-02-18, capacity=5: только 1 positive, 0 negative
  Пропущена группа 2026-02-09, capacity=3: только 2 positive, 0 negative
  Пропущена группа 2026-02-09, capacity=5: только 0 positive, 2 negative
  Пропущена группа 2026-02-22, capacity=3: только 0 positive, 2 negative
  Пропущена группа 2026-02-22, capacity=5: только 1 positive, 0 negative
  Пропущена группа 2026-02-28, capacity=1: только 4 positive, 0 negative
  Пропущена группа 2026-02-28, capacity=4: только 8 positive, 0 negative
  Пропущена группа 2026-02-21, capacity=1: только 1 positive, 0 negative
  Пропущена группа 2026-02-15, capacity=4: только 3 positive, 0 negative
  Пропущена группа 2026-02-24, capacity=1: только 2 positive, 0 negative
  Пропущена группа 2026-02-16, capacity=4: только 3 positive, 0 negative
  Пропущена группа 2026-02-27, capacity=1: только 